In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:  # aqui mantemos menor que o limite
            return True
        return False
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.8)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_5_1_8,0.798364,-0.351033,-0.083071,0.821900,0.641102,0.295258,1.978337,0.938213,0.202033,0.570123,3.147670,0.543377,0.308677,0.566509,36.439810,57.160699,"Hidden Size=[4], regularizer=0.2, learning_rat..."
1,model_5_1_5,0.798151,-0.330215,0.150646,0.784124,0.691338,0.295570,1.947853,0.735755,0.244886,0.490320,2.698866,0.543664,0.307947,0.566809,36.437697,57.158586,"Hidden Size=[4], regularizer=0.2, learning_rat..."
2,model_5_1_9,0.797605,-0.319013,-0.085584,0.812757,0.637152,0.296370,1.931450,0.940390,0.212405,0.576398,2.542732,0.544398,0.306075,0.567575,36.432295,57.153184,"Hidden Size=[4], regularizer=0.2, learning_rat..."
4,model_5_1_6,0.797170,-0.351358,0.093542,0.777424,0.673376,0.297007,1.978813,0.785222,0.252486,0.518854,2.306323,0.544984,0.304582,0.568185,36.427997,57.148886,"Hidden Size=[4], regularizer=0.2, learning_rat..."
5,model_5_1_4,0.796623,-0.313465,0.189901,0.790595,0.704352,0.297808,1.923326,0.701750,0.237545,0.469647,2.946287,0.545718,0.302707,0.568950,36.422612,57.143501,"Hidden Size=[4], regularizer=0.2, learning_rat..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
519,model_13_4_11,0.752760,0.457956,0.322705,0.929665,0.780825,0.673974,1.477607,1.206207,0.334823,0.770515,1.524488,0.820959,6.933759,0.855909,50.789126,81.261022,"Hidden Size=[6], regularizer=0.2, learning_rat..."
520,model_31_3_11,0.752664,0.371327,0.797058,0.225522,0.768767,0.674235,1.713758,0.526066,0.814364,0.670215,0.831849,0.821118,1.282669,0.856075,90.788353,145.637765,"Hidden Size=[11], regularizer=0.05, learning_r..."
521,model_29_4_8,0.752637,0.436386,0.487680,0.496475,0.541318,0.674311,1.536407,1.402371,1.593904,1.498138,1.098186,0.821164,1.282701,0.856123,90.788127,145.637539,"Hidden Size=[11], regularizer=0.2, learning_ra..."
523,model_23_1_13,0.752564,-0.008043,0.661217,0.951246,0.849250,0.362325,1.476092,0.314684,0.067214,0.190949,2.098851,0.601934,1.014736,0.627560,856.030428,1376.490405,"Hidden Size=[14, 24], regularizer=0.05, learni..."
